# Modify
Normalización, comprobación de particiones y representación numérica compartida.

In [1]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next(folder for folder in (cwd, *cwd.parents)
            if (folder / "data" / "dataset.csv").is_file()
            and (folder / "scripts" / "corpus.py").is_file())
sys.path.insert(0, str(ROOT))

if "show_chart" not in globals():
    def show_chart(name):
        from IPython.display import SVG, display
        display(SVG(filename=str(ROOT / "artifacts" / "charts" / f"{name}.svg")))


In [2]:
from scripts import corpus
from scripts.train import vocabulary_for
from vulnminer.model import vectorize
rows = corpus.normalize(corpus.read())
train = [row for row in rows if row["partition"] == "train"]
vocabulary = vocabulary_for(train)
vectors = [vectorize(row["features"], vocabulary) for row in rows]
print("Dimensiones TF-IDF", len(vectors), len(vocabulary))
print("Media de rasgos distintos por fila", round(sum(map(len, vectors)) / len(vectors), 2))
print("Ejemplo de rasgos y pesos")
for item, weight in list(vectors[0].items())[:8]:
    print(item, round(weight, 4))

Dimensiones TF-IDF 800 404
Media de rasgos distintos por fila 13.0
Ejemplo de rasgos y pesos
a:object=true 0.2865
a:orders=true 0.3171
a:owner=true 0.2377
a:verify=true 0.2377
b:object/owner 0.2377
b:orders/verify 0.4136
b:owner/true 0.2865
b:verify/object 0.2865


In [3]:
projections = {}
for row, vector in zip(rows, vectors):
    projections.setdefault(row["partition"], set()).add(tuple(vector))
for name, values in projections.items():
    print(name, "vectores distintos", len(values))
for name in ("validation", "test"):
    print(name, "vectores iguales a train", len(projections[name] & projections["train"]))

train vectores distintos 560
validation vectores distintos 40
test vectores distintos 40
validation vectores iguales a train 0
test vectores iguales a train 0


La extracción conserva tokens, pares vecinos y asociaciones clave–estado normalizadas. El vocabulario y la ponderación se ajustan solo sobre entrenamiento. La regresión usa TF-IDF con norma L2 y pondera el desbalance uno contra resto. Naive Bayes usa presencia binaria y suavizado. El entrenamiento exporta la matriz del modelo seleccionado a `artifacts/features.csv`.